In [1]:
import requests
import pandas as pd

documents = requests.get(
    'https://datatalks.club/faq/json/data-engineering-zoomcamp.json'
).json()

df = pd.DataFrame(documents)
df = df.rename(columns={'answer': 'text'})
df.head()

,id,course,section,question,text
0,9e508f2212,data-engineering-zoomcamp,General Course-Related Questions,Course: When does the course start?,A new cohort runs roughly January–April every ...
1,bfafa427b3,data-engineering-zoomcamp,General Course-Related Questions,Course: What are the prerequisites for this co...,"To get the most out of this course, you should..."
2,3f1424af17,data-engineering-zoomcamp,General Course-Related Questions,Course: Can I still join the course after the ...,"Yes, even if you don't register, you're still ..."
3,52217fc51b,data-engineering-zoomcamp,General Course-Related Questions,Course: I have registered for the Data Enginee...,You don't need a confirmation email. You're ac...
4,33fc260cd8,data-engineering-zoomcamp,General Course-Related Questions,Course: What can I do before the course starts?,Get the basic environment ready ahead of time:...


In [2]:
print(f"Total de documentos: {len(df)}")
print(f"\nSecciones únicas:")
print(df['section'].value_counts())

Total de documentos: 402

Secciones únicas:
section
Module 4: Analytics Engineering with dbt          68
Module 3: Data Warehousing                        48
Module 6: Spark                                   35
Module 1: Docker                                  33
Module 7: Streaming                               32
General Course-Related Questions                  26
Module 2: Workflow Orchestration                  26
Module 1: Postgres, pgAdmin & Python ingestion    25
Module 1: GCP setup & VM                          24
Environment & Setup                               23
Workshop 1 - dlthub                               18
Module 1: Terraform                               14
Project                                           12
Module 5: Data Platforms (Bruin)                  10
Module 1: Taxi Data (download & handling)          8
Name: count, dtype: int64


In [3]:
docs_example = [
    "Course starts on 15th Jan 2024",
    "Prerequisites listed on GitHub",
    "Submit homeworks after start date",
    "Registration not required for participation",
    "Setup Google Cloud and Python before course",
]

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')
X = cv.fit_transform(docs_example)

names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names).T
df_docs

,0,1,2,3,4
15th,1,0,0,0,0
2024,1,0,0,0,0
cloud,0,0,0,0,1
course,1,0,0,0,1
date,0,0,1,0,0
github,0,1,0,0,0
google,0,0,0,0,1
homeworks,0,0,1,0,0
jan,1,0,0,0,0
listed,0,1,0,0,0


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

cv = TfidfVectorizer(stop_words='english')
X = cv.fit_transform(docs_example)

names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names).T
df_docs.round(2)

,0,1,2,3,4
15th,0.46,0.00,0.0,0.00,0.00
2024,0.46,0.00,0.0,0.00,0.00
cloud,0.00,0.00,0.0,0.00,0.46
course,0.37,0.00,0.0,0.00,0.37
date,0.00,0.00,0.5,0.00,0.00
github,0.00,0.58,0.0,0.00,0.00
google,0.00,0.00,0.0,0.00,0.46
homeworks,0.00,0.00,0.5,0.00,0.00
jan,0.46,0.00,0.0,0.00,0.00
listed,0.00,0.58,0.0,0.00,0.00


In [6]:
query = "Do I need to know python to sign up for the January course?"

q = cv.transform([query])
q.toarray()

array([[0.        , 0.        , 0.        , 0.62791376, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.77828292, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ]])

In [7]:
cv.vocabulary_

{'course': 3,
 'starts': 17,
 '15th': 0,
 'jan': 8,
 '2024': 1,
 'prerequisites': 11,
 'listed': 9,
 'github': 5,
 'submit': 18,
 'homeworks': 7,
 'start': 16,
 'date': 4,
 'registration': 13,
 'required': 14,
 'participation': 10,
 'setup': 15,
 'google': 6,
 'cloud': 2,
 'python': 12}

In [8]:
X.dot(q.T).toarray()

array([[0.23490553],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.59579005]])

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(X, q)

array([[0.23490553],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.59579005]])

In [10]:
fields = ['section', 'question', 'text']
transformers = {}
matrices = {}

for field in fields:
    cv = TfidfVectorizer(stop_words='english', min_df=3)
    X = cv.fit_transform(df[field])

    transformers[field] = cv
    matrices[field] = X

In [11]:
for field in fields:
    print(f"{field}: {len(transformers[field].vocabulary_)} palabras")

section: 32 palabras
question: 252 palabras
text: 1352 palabras


In [12]:
print("Tamaño de las matrices generadas (Documentos, Palabras en el vocabulario):")
for field in fields:
    print(f"{field}: {matrices[field].shape}")

Tamaño de las matrices generadas (Documentos, Palabras en el vocabulario):
section: (402, 32)
question: (402, 252)
text: (402, 1352)


In [13]:
query = "I just signed up. Is it too late to join the course?"

q = transformers['text'].transform([query])
score = cosine_similarity(matrices['text'], q).flatten()

In [14]:
import numpy as np

mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

In [15]:
idx = np.argsort(-score)[:10]
df.iloc[idx].text

13     No, as long as you complete the peer-reviewed ...
0      A new cohort runs roughly January–April every ...
16     No, late submissions are not allowed. However,...
307    The latest filename is just `taxi_zone_lookup....
22     You will have two attempts for a project.\n\n-...
24     There will be two announcements in the course ...
262    This error could result if you are using a `SE...
3      You don't need a confirmation email. You're ac...
99     In join queries, if you mention the column nam...
15     Deadlines are published per cohort. Find them ...
Name: text, dtype: str

In [18]:
query = "I just signed up. Is it too late to join the course?"

boost = {'question': 3.0}

score = np.zeros(len(df))

for f in fields:
    b = boost.get(f, 1.0)
    q = transformers[f].transform([query])
    s = cosine_similarity(matrices[f], q).flatten()
    score = score + b * s

In [17]:
# 1. Aplicamos el filtro duro (Filtering) para DE Zoomcamp
mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

# 2. Ordenamos descendentemente y tomamos los 5 índices más altos
idx = np.argsort(-score)[:5]

# 3. Imprimimos para ver el efecto del Boosting
print("Top 5 resultados con Boosting en 'question':\n")
for i, index in enumerate(idx):
    print(f"--- Rango {i+1} (Puntaje: {score[index]:.3f}) ---")
    print(f"Pregunta original: {df.iloc[index]['question']}")
    print(f"Texto: {df.iloc[index]['text'][:100]}...\n")

Top 5 resultados con Boosting en 'question':

--- Rango 1 (Puntaje: 3.587) ---
Pregunta original: Edit Course Profile.
Texto: The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to ...

--- Rango 2 (Puntaje: 3.543) ---
Pregunta original: Course: What are the prerequisites for this course?
Texto: To get the most out of this course, you should have:

- Basic coding experience
- Familiarity with S...

--- Rango 3 (Puntaje: 3.500) ---
Pregunta original: Course: What can I do before the course starts?
Texto: Get the basic environment ready ahead of time:

- Google Cloud account (free trial — see the GCP set...

--- Rango 4 (Puntaje: 3.500) ---
Pregunta original: How can we contribute to the course?
Texto: - [Star the repository](https://github.com/DataTalksClub/data-engineering-zoomcamp).
- Share it with...

--- Rango 5 (Puntaje: 3.187) ---
Pregunta original: Course: When does the course start?
Texto: A new cohort runs roughly January–April every

In [20]:
filters = {
    'course': 'data-engineering-zoomcamp',
}

for field, value in filters.items():
    mask = (df[field] == value).values
    score = score * mask

In [21]:
idx = np.argsort(-score)[:10]
results = df.iloc[idx]
results.to_dict(orient='records')

[{'id': '16005581f2',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Edit Course Profile.',
  'text': 'The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if you prefer. Your entry on the Leaderboard is the one highlighted in light green.\n\nThe Certificate name should be your actual name that you want to appear on your certificate after completing the course.\n\nThe "Display on Leaderboard" option indicates whether you want your name to be listed on the course leaderboard.'},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'text': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering 

In [22]:
class TextSearch:

    def __init__(self, text_fields):
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorizers = {}

    def fit(self, records, vectorizer_params={}):
        self.df = pd.DataFrame(records)
        if 'answer' in self.df.columns:
            self.df = self.df.rename(columns={'answer': 'text'})

        for f in self.text_fields:
            cv = TfidfVectorizer(**vectorizer_params)
            X = cv.fit_transform(self.df[f])
            self.matrices[f] = X
            self.vectorizers[f] = cv

    def search(self, query, n_results=10, boost={}, filters={}):
        score = np.zeros(len(self.df))

        for f in self.text_fields:
            b = boost.get(f, 1.0)
            q = self.vectorizers[f].transform([query])
            s = cosine_similarity(self.matrices[f], q).flatten()
            score = score + b * s

        for field, value in filters.items():
            mask = (self.df[field] == value).values
            score = score * mask

        idx = np.argsort(-score)[:n_results]
        results = self.df.iloc[idx]
        return results.to_dict(orient='records')

In [23]:
# 1. Instanciamos el motor indicándole los campos de búsqueda
index = TextSearch(
    text_fields=['section', 'question', 'text']
)

# 2. Entrenamos el motor (Asegúrate de pasarle los datos correctos)
# El workshop asume que 'documents' es la lista cruda de diccionarios que descargaste el primer día
index.fit(documents, vectorizer_params={'stop_words': 'english', 'min_df': 3})

# 3. Ejecutamos una búsqueda de producción
results = index.search(
    query='I just signed up. Is it too late to join the course?',
    n_results=5,
    boost={'question': 3.0},
    filters={'course': 'data-engineering-zoomcamp'}
)

print(results)

[{'id': '16005581f2', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Edit Course Profile.', 'text': 'The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if you prefer. Your entry on the Leaderboard is the one highlighted in light green.\n\nThe Certificate name should be your actual name that you want to appear on your certificate after completing the course.\n\nThe "Display on Leaderboard" option indicates whether you want your name to be listed on the course leaderboard.'}, {'id': 'bfafa427b3', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: What are the prerequisites for this course?', 'text': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is nec